# Weather Project
This project is an end-to-end machine learning (ML) application that predicts tomorrow's temperature and weather, based on London's historical data, between 1979 and 2021.

The dataset contains over 15,000+ observations with features: cloud, cloud_cover, sunshine, global radiation, temperature, precipitation, pressure, and snow depth.

This project involves data cleaning, exploratory data analysis (EDA) using interactive visualisations, feature engineering, model training (regression/classification) and evaluation. The final models will be integrated into a fully interactive Streamlit web application.

***


## 1. Loading Data

In [22]:
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime

In [23]:
df = pd.read_csv("london_weather.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

Shape: (15341, 10)
Columns: Index(['date', 'cloud_cover', 'sunshine', 'global_radiation', 'max_temp',
       'mean_temp', 'min_temp', 'precipitation', 'pressure', 'snow_depth'],
      dtype='object')


## 2. Cleaning Data

In [24]:
df.isna().sum()


date                   0
cloud_cover           19
sunshine               0
global_radiation      19
max_temp               6
mean_temp             36
min_temp               2
precipitation          6
pressure               4
snow_depth          1441
dtype: int64

In [25]:
df = df.dropna(how="all")

In [26]:
df["snow_depth"] = df["snow_depth"].fillna(0)
df["cloud_cover"] = df["cloud_cover"].fillna(0)
df["precipitation"] = df["precipitation"].fillna("0")

In [27]:
df = df.dropna(subset=["min_temp", "max_temp", "max_temp", "pressure", "global_radiation"])

df["mean_temp"] = df["mean_temp"].fillna((df["min_temp"] + df["max_temp"] ) / 2) 

In [28]:
df["global_radiation"] = df["global_radiation"].fillna(method="ffill")
df["max_temp"] = df["max_temp"].fillna(method="ffill")
df["mean_temp"] = df["mean_temp"].fillna(method="ffill")
df["min_temp"] = df["min_temp"].fillna(method="ffill")


C:\Users\markn\AppData\Local\Temp\ipykernel_9024\1530177353.py:1: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\markn\AppData\Local\Temp\ipykernel_9024\1530177353.py:2: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\markn\AppData\Local\Temp\ipykernel_9024\1530177353.py:3: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\markn\AppData\Local\Temp\ipykernel_9024\1530177353.py:4: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [29]:
df[df["max_temp"] > 50]

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth


In [30]:
df.head(6)

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth
0,19790101,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0
1,19790102,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0
2,19790103,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0
3,19790104,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0
4,19790105,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0
5,19790106,5.0,3.8,39.0,8.3,-0.5,-6.6,0.7,102780.0,1.0


In [31]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

In [32]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0


In [33]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day_of_year

In [34]:
winter = [12, 1, 2]
spring = [3, 4, 5]
summer = [6, 7, 8]
autumn = [9, 10 , 11]

def get_season(month):
    if month in winter:
        return "winter"
    elif month in spring:
        return "spring"
    elif month in autumn:
        return "autumn"
    else:
        return "summer"
    
df["season"] = df["month"].apply(get_season)

In [35]:
df.head(6)

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month,day,season
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1,1,winter
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1,2,winter
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1,3,winter
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1,4,winter
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1,5,winter
5,1979-01-06,5.0,3.8,39.0,8.3,-0.5,-6.6,0.7,102780.0,1.0,1979,1,6,winter


In [36]:
df.columns

Index(['date', 'cloud_cover', 'sunshine', 'global_radiation', 'max_temp',
       'mean_temp', 'min_temp', 'precipitation', 'pressure', 'snow_depth',
       'year', 'month', 'day', 'season'],
      dtype='object')

## 3. Exploratory Data Analysis (EDA)

In [37]:
px.scatter(df, x="date", y="mean_temp",
           title="Mean Temprerature Over Time (1979 - 2021)", 
           opacity=0.4,
           color="season", 
           labels={"mean_temp": "Mean Temperature (C°)", "date": "Date (Year)"})


This plot shows the long term temprature trend in London from 1979 to 2021. We can see that there are month / season based fluctuations in temprature, with summer months being warmer and winter months being colder. However, there is also a slight upward trend in temperature over the years, indicating that London is getting warmer, likely due to climate change.

In [38]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month,day,season
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1,1,winter
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1,2,winter
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1,3,winter
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1,4,winter
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1,5,winter


In [48]:
fig = px.line(df, x="day", y="pressure", animation_frame="year", title="Animated Daily Pressure Curve by Year", labels={"pressure": "Pressure (Pa)", "day": "Day of the Year"})
fig.show()

In [40]:
fig.show()